# 03 — Entorno de entrenamiento y linea base ResNet

Cubre las tareas **4.1** (entorno de entrenamiento) y **4.2** (arquitectura
residual) del WBS. El alcance completo del bloque esta en
[`docs/modelado.md`](../docs/modelado.md).

Lo que hace esta notebook, en orden:

1. Baja el dataset etiquetado desde Hugging Face.
2. Parte los datos **por partida**, no por posicion.
3. Precomputa el cache de tensores.
4. Mide los dos baselines no neuronales, que fijan el piso de RMSE.
5. Construye la ResNet y verifica que aprende.

> **Runtime: GPU (T4).** Al reves que las notebooks del pipeline de datos, que
> pedian CPU. Un modelo de ~3 M de parametros sobre entradas de 8x8 no
> justifica una A100: quema unidades ~6 veces mas rapido y queda
> desaprovechada.


## 1. Entorno


In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Modelo de GPU  :", torch.cuda.get_device_name(0))
print("CPU workers    :", runtime.cpu_count)
print("torch          :", torch.__version__)
for aviso in runtime.warnings():
    print("AVISO:", aviso)

## 2. Tests

Conviene correrlos antes de gastar cuota de GPU. La salida sirve ademas como
evidencia de los requerimientos de testing (3.1 y 3.2) para la memoria.


In [ ]:
!{sys.executable} -m pytest -q

## 3. Cargar el dataset


In [ ]:
from chessdl.config import load_config
from chessdl import hf
from chessdl.data import schema

cfg = load_config()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))
print(f"{tabla.num_rows:,} posiciones")

In [ ]:
import numpy as np

# Solo las columnas que necesita el entrenamiento: el resto no entra en RAM.
fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
# El target es value_stm: la etiqueta en perspectiva del jugador al turno,
# que es la que corresponde a la entrada espejada.
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)
print(f"{len(fens):,} posiciones de {len(set(game_ids)):,} partidas")

## 4. Particion por partida

**La decision mas importante del bloque.** Hay 4 posiciones por partida, y
las de una misma partida comparten apertura, jugadores y estructura. Partir
por posicion deja hermanas a ambos lados del muro: la validacion queda
inflada y el error recien aparece cuando el motor juega peor que lo que
prometian las metricas.

La asignacion es por *hash* del `game_id`, no por mezcla de la lista. Asi es
**estable si el dataset crece**: agregar shards mañana deja cada partida en
el split que ya tenia.


In [ ]:
from chessdl.training.split import describe_split, leaked_games, split_masks

split_cfg = cfg.training.split_config()
masks = split_masks(game_ids, split_cfg)

print(describe_split(masks, game_ids))
print()
fugadas = leaked_games(masks, game_ids)
print("Partidas en mas de un split:", len(fugadas), "(tiene que ser 0)")
assert not fugadas

## 5. Cache de tensores

Codificar un FEN cuesta ~79 microsegundos, de los cuales 44 son parsear el
FEN. Con 2 vCPU son ~100 segundos por epoca solo codificando, contra ~140
que tarda la T4: sin cache el tiempo por epoca casi se duplica y la mitad de
las unidades se va en esperar a la CPU.

El cache son 2,9 GB en `uint8` y se construye una sola vez. **No modifica el
dataset**: el Parquet sigue guardando FEN.


In [ ]:
from chessdl.training.cache import build_cache, cache_path_for, cache_size_bytes, load_cache

ruta_cache = cache_path_for(cfg.training.cache_dir)
print(f"Tamano estimado: {cache_size_bytes(len(fens))/1e9:.1f} GB -> {ruta_cache}")

if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)

cache = load_cache(ruta_cache, expected_rows=len(fens))
print("Cache listo:", cache.shape, cache.dtype)

In [ ]:
# El cache tiene que ser indistinguible del encoder: si no, el modelo se
# entrena con una representacion y el motor corre con otra, sin que falle nada.
import chess
from chessdl.encoding import board_to_tensor
from chessdl.training.cache import cached_to_tensor

rng = np.random.default_rng(0)
iguales = 0
for i in rng.choice(len(fens), size=500, replace=False):
    directo = board_to_tensor(chess.Board(fens[i]))
    iguales += np.array_equal(cached_to_tensor(np.asarray(cache[i])), directo)
print(f"Posiciones verificadas: 500   identicas al encoder: {iguales}")
assert iguales == 500

## 6. Baselines no neuronales

Un RMSE suelto no se puede interpretar. Estos dos lo acotan:

- **Media constante** — el piso absoluto. Un modelo que no le gane tiene un
  error de implementacion, no un problema de arquitectura.
- **Material lineal** — el piso "sabe algo de ajedrez". La diferencia entre
  este y la red es lo que la red realmente aporta.


In [ ]:
from chessdl.training.baselines import describe_weights, material_baseline, mean_baseline

idx_train = np.flatnonzero(masks['train'])
idx_val   = np.flatnonzero(masks['val'])

# Submuestra para el ajuste lineal: con 2,3 M de filas no hace falta mas.
sub_train = rng.choice(idx_train, size=min(200_000, len(idx_train)), replace=False)

media = mean_baseline(targets[idx_train], targets[idx_val])
material, pesos = material_baseline(
    np.asarray(cache[np.sort(sub_train)]), targets[np.sort(sub_train)],
    np.asarray(cache[idx_val]), targets[idx_val],
)
print(media)
print(material)

In [ ]:
# Los pesos deberian parecerse a los valores de manual. Si no, sospechar
# de la codificacion antes que del modelo.
print(describe_weights(pesos))

## 7. La ResNet (tarea 4.2)

Implementada desde cero. Una ResNet de `torchvision` arranca con convolucion
7x7 de stride 2 y max-pooling: dejaria el tablero de 8x8 en 2x2 en dos pasos.
Aca la convolucion es 3x3 con padding 1, que mantiene 8x8 de punta a punta.

La salida pasa por `tanh`, asi el rango del **requerimiento 1.4** queda
garantizado por construccion y no por lo que la red haya aprendido.


In [ ]:
from chessdl.models.resnet import ChessResNet, ResNetConfig

modelo = ChessResNet(ResNetConfig(channels=128, blocks=8))
print(modelo.describe())
print()
print(modelo)

In [ ]:
# Comprobacion de cordura: el rango de salida y que el tablero no se reduzca.
modelo.eval()
with torch.no_grad():
    salida = modelo(torch.randn(256, 18, 8, 8) * 100)
print(f"Salida con entradas extremas: [{salida.min():+.4f}, {salida.max():+.4f}]  (requerimiento 1.4: [-1, 1])")

with torch.no_grad():
    cuerpo = modelo.blocks(modelo.stem(torch.zeros(2, 18, 8, 8)))
print("El tablero sigue siendo 8x8 tras el cuerpo:", tuple(cuerpo.shape))

### Que la red puede aprender

Antes de lanzar una campana completa, la prueba estandar: sobreajustar un
lote chico. Si no puede llevar la perdida a cero sobre 512 posiciones fijas,
el problema es estructural y ningun ajuste de hiperparametros lo arregla.


In [ ]:
from torch.utils.data import DataLoader
from chessdl.training.dataset import PositionDataset

torch.manual_seed(0)
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

lote_idx = np.sort(rng.choice(idx_train, size=512, replace=False))
mini = PositionDataset(targets, cache=cache, indices=lote_idx)
xb = torch.stack([mini[i][0] for i in range(len(mini))]).to(dispositivo)
yb = torch.tensor(mini.targets).to(dispositivo)

prueba = ChessResNet(ResNetConfig(channels=64, blocks=4)).to(dispositivo).train()
opt = torch.optim.Adam(prueba.parameters(), lr=3e-4)
for paso in range(300):
    opt.zero_grad()
    perdida = torch.nn.functional.mse_loss(prueba(xb), yb)
    perdida.backward(); opt.step()
    if paso % 50 == 0:
        print(f"  paso {paso:3d}: {perdida.item():.6f}")
print(f"\nPerdida final: {perdida.item():.6f}   (varianza de las etiquetas: {yb.var().item():.4f})")

## 8. Resumen

Lo que queda listo para la primera campana de entrenamiento (tarea 4.4).


In [ ]:
print(f"{'Posiciones':<34}{tabla.num_rows:>14,}")
print(f"{'Particion (train/val/test)':<34}{str(split_cfg.train)+'/'+str(split_cfg.val)+'/'+str(split_cfg.test):>14}")
print(f"{'Semilla de particion':<34}{split_cfg.seed:>14,}")
print(f"{'Cache de tensores':<34}{str(cache.shape):>14}")
print(f"{'Parametros de la ResNet':<34}{modelo.count_parameters():>14,}")
print()
print(f"{'RMSE - media constante':<34}{media.rmse:>14.4f}")
print(f"{'RMSE - material lineal':<34}{material.rmse:>14.4f}")
print()
print("La red tiene que quedar por debajo de esos dos numeros.")